In [1]:
from utils.utils import load_from_pickle

testset = load_from_pickle('testset')
dataset = testset.to_pandas()

In [2]:
dataset2 = dataset[:2]

In [3]:
dataset2.head(1)

,question,contexts,ground_truth,evolution_type,metadata,episode_done
0,What challenges does the concept of kearifan l...,[DOC NAME: Undang-undang Nomor 15 Tahun 2023 T...,The concept of kearifan lokal in Bali faces ch...,simple,[{'title': 'Undang-undang Nomor 15 Tahun 2023 ...,True


In [4]:
questions = list(dataset2["question"])
questions

["What challenges does the concept of kearifan lokal face in the context of Bali's cultural development?",
 'What specific responsibilities does Lembaga Penjamin Simpanan (LPS) have under Undang-undang Nomor 4 Tahun 2023 regarding the management and resolution of banks and insurance companies that have had their licenses revoked by Otoritas Jasa Keuangan?']

In [5]:
from utils.llm import get_llm, get_embedding_model
from utils.database.qdrant import QdrantIndexManager
from utils.retriever.uu.embedding_uu import get_embedding_uu
from utils.retriever.uu.graph_uu import get_graph_uu
from utils.config import get_config
from utils.chain.rag import create_combined_answer_chain


llm = get_llm("llama3.1")
embed_model = get_embedding_model("BAAI/bge-m3")
vector_store = QdrantIndexManager(embed_model=embed_model, index_name="uu").load_vector_index()
retriever_uu = get_embedding_uu(vector_store=vector_store, llm=llm, embed_model=embed_model, with_self_query=False)

config = get_config()
graph_chain = get_graph_uu(llm=llm, config=config)

chain_quality = create_combined_answer_chain(
  retriever_uu=retriever_uu,
  graph_chain=graph_chain,
  llm=llm
)

Loaded index 'uu'.


In [6]:
result = chain_quality.invoke({"question": "Apa ibukota dari Provinsi Jawa Barat?", "chat_history": ""})
result

{'rewrited question': 'Apa ibukota dari Provinsi Jawa Barat?',
 'context': [Document(metadata={'title': 'Undang-undang Nomor 10 Tahun 2023 Tentang Provinsi Jawa Barat', 'jenis_bentuk_peraturan': 'UNDANG-UNDANG', 'pemrakarsa': 'PEMERINTAH PUSAT', 'nomor': 10, 'tahun': 2023, 'tentang': 'PROVINSI JAWA BARAT', 'tempat_penetapan': 'Jakarta', 'ditetapkan_tanggal': '04 Mei 2023', 'status': 'Berlaku', 'dokumen_peraturan': 'https://peraturan.go.id/files/uu-no-10-tahun-2023.pdf', 'id': 'hqel0eiu', 'filename': 'uu_04_mei_2023_uu_no_10_tahun_2023_hqel0eiu.pdf', 'page_number': 3, '_id': '58f2f3c8-d6c5-4fed-98c1-b98efa75969f', '_collection_name': 'uu'}, page_content='- 3 -\nBAB II\nCAKUPAN WILAYAH, IBU KOTA, DAN KARAKTERISTIK\nPROVINSI JAWA BARAT\nPasal 3\nProvinsi Jawa Barat terdiri atas 18 (delapan belas).\nkabupaten dan 9 (sembilan) kota, yaitu:.\n Kabupaten Bogor;\nb.\nKabupaten Sukabumi;\nKabupaten Cianjur;\nd. Kabupaten Bandung;\ne.\nKabupaten Garut;\nf.\nKabupaten Tasikmalaya;\nKabupaten Ciam

In [7]:
ctx = result["context"]
len(ctx)

6

In [8]:
def unique_context(context: list):
    context_unique = [i for n, i in enumerate(context) if i not in context[:n]]
    return context_unique

ctx_unique = unique_context(ctx)
len(ctx_unique)

6

# Chain

In [9]:
answer = []
context = []

In [10]:
import warnings
warnings.filterwarnings("ignore")

In [11]:
result2 = chain_quality.invoke({"question":questions[1], "chat_history":"-"})

In [12]:
i = 1

for question in questions:
    try:
        result = chain_quality.invoke({"question":question, "chat_history":"-"})
        print("Successfully run the chain for question", i)
        answer.append(result['answer'])
        context.append(result['context'])
        i += 1
    
    except Exception:
        print("Failed run the chain for question", i)
        break

Successfully run the chain for question 1
Successfully run the chain for question 2


In [13]:
context_unique = [unique_context(ctx) for ctx in context]
context_unique

[[Document(metadata={'title': 'Undang-undang Nomor 15 Tahun 2023 Tentang Provinsi Bali', 'jenis_bentuk_peraturan': 'UNDANG-UNDANG', 'pemrakarsa': 'PEMERINTAH PUSAT', 'nomor': 15, 'tahun': 2023, 'tentang': 'PROVINSI BALI', 'tempat_penetapan': 'Jakarta', 'ditetapkan_tanggal': '04 Mei 2023', 'status': 'Berlaku', 'dokumen_peraturan': 'https://peraturan.go.id/files/uu-no-15-tahun-2023.pdf', 'id': 'pvfm3kfq', 'filename': 'uu_04_mei_2023_uu_no_15_tahun_2023_pvfm3kfq.pdf', 'page_number': 9, '_id': '706e9857-c1d5-4f4b-954f-e11d4bc84fb1', '_collection_name': 'uu'}, page_content='- 2 -\nPermasalahan kebudayaan yang dihadapi Provinsi Bali mencakup\nadat istiadat, tradisi, seni dan budaya serta kearifan lokal yang terus\nmengalami kemunduran dari segi jumlah dan kualitas kelembagaan,\nsarana dan prasarana, sumber daya manusia serta sistem nilai dan\npranata budaya. Penyelenggaraan pembangunan Bali juga belum\nsepenuhnya menjamin keajekan nilai budaya, adat istiadat, aura (taksu)\nBali, dan kearifan

In [14]:
ground_truth = list(dataset2["ground_truth"])
ground_truth

['The concept of kearifan lokal in Bali faces challenges such as a decline in the number and quality of institutions, facilities, human resources, and the value systems and cultural institutions. Additionally, the development in Bali has not fully guaranteed the preservation of cultural values, traditions, and local wisdom as the identity of the Balinese people.',
 'Lembaga Penjamin Simpanan (LPS) has specific responsibilities under Undang-undang Nomor 4 Tahun 2023 regarding the management and resolution of banks and insurance companies that have had their licenses revoked by Otoritas Jasa Keuangan, which include: 1. Managing and handling the Bank Dalam Resolusi and insurance companies that are being addressed and resolved by LPS; 2. Managing the assets of the Bank Dalam Resolusi and insurance companies that are handed over for handling or resolution to LPS; and 3. Settling the assets and liabilities of the Bank Dalam Resolusi and insurance companies whose licenses have been revoked by

In [15]:
context_unique_str = []

for i in context_unique:
    ls = []
    for j in i:
        ls.append(j.page_content)
    context_unique_str.append(ls)
    
context_unique_str

[['- 2 -\nPermasalahan kebudayaan yang dihadapi Provinsi Bali mencakup\nadat istiadat, tradisi, seni dan budaya serta kearifan lokal yang terus\nmengalami kemunduran dari segi jumlah dan kualitas kelembagaan,\nsarana dan prasarana, sumber daya manusia serta sistem nilai dan\npranata budaya. Penyelenggaraan pembangunan Bali juga belum\nsepenuhnya menjamin keajekan nilai budaya, adat istiadat, aura (taksu)\nBali, dan kearifan lokal sebagai jati diri masyarakat Bali. Hal ini\nberpihak pada pemajuan kebudayaan Bali.\nBerkaitan dengan hal di atas, Undang-Undang ini juga dibentuk\nPembentukan Daerah-daerah tingkat I Bali, Nusa Tenggara Barat dan Nusa\nTenggara Timur yang memuat penyempurnaan dasar hukum, penyesuaian\ncakupan wilayah, pengakuan karakteristik Provinsi Bali serta kontribusi\nkebudayaan dan desa adat di Bali.\nII. PASAL DEMI PASAL\nPasal 1\nCukup jelas.\nPasal 2\nCukup jelas.\nPasal 3\nCukup jelas.\nPasal 4\nCukup jelas.\nPasal 5\nCukup jelas.\nPasal 6.\nSK No 181554 A\n\nSource

In [16]:
len(context_unique_str[0])

8

In [17]:
from datasets import Dataset

dict_eval = {
    "question": questions,
    "answer": answer,
    "contexts": context_unique_str,
    "ground_truth": ground_truth
}

dataset = Dataset.from_dict(dict_eval)

In [18]:
import nest_asyncio
import warnings

nest_asyncio.apply()
warnings.filterwarnings("ignore")

In [19]:
from utils.llm import get_llm, get_embedding_model

llm = get_llm("llama3.1")
embed_model = get_embedding_model("BAAI/bge-m3")

In [26]:
from ragas.metrics import FaithulnesswithHHEM, answer_relevancy, context_precision, context_recall, answer_similarity, answer_correctness
from ragas.metrics.critique import harmfulness
from ragas import evaluate

faithfulness_with_hhem = FaithulnesswithHHEM()

metrics = [faithfulness_with_hhem, answer_relevancy, context_precision, context_recall]

You are using a model of type HHEMv2Config to instantiate a model of type HHEMv2. This is not supported for all configurations of models and can yield errors.


In [27]:
score = evaluate(dataset, metrics, llm, embed_model)

Evaluating:   0%|          | 0/4 [00:00<?, ?it/s]

Token indices sequence length is longer than the specified maximum sequence length for this model (3621 > 512). Running this sequence through the model will result in indexing errors
Exception raised in Job[0]: TimeoutError()
Exception raised in Job[3]: TimeoutError()
Exception raised in Job[1]: TimeoutError()


In [28]:
score

{'faithfulness_with_hhem': 0.0000, 'answer_relevancy': nan}